# Stage 1 Checkpoint Evaluation

Evaluation-only extension of notebook 6. This compares saved Stage 1A and Stage 1B autoencoder checkpoints on fixed held-out test splits and writes the requested selection tables, per-example metrics, split fingerprints, plots, and final Stage 2 checkpoint manifest.

No training occurs in this notebook.

In [ ]:
from pathlib import Path
import os
import sys

REPO_DIR = Path.cwd()
if not (REPO_DIR / "experiments/3dcnn").exists():
    REPO_DIR = Path("/Users/borng/code/lab_work/neurovlm")
sys.path.insert(0, str(REPO_DIR / "experiments/3dcnn"))

from stage1_checkpoint_evaluation import EvaluationConfig, run_evaluation

# Set this to the completed notebook-6 ablation run directory.
# Example: /content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_ae_ablation/ae_ablation_20260623_123456
RUN_ROOT = Path(os.environ.get("NEUROVLM_AE_ABLATION_RUN_DIR", "")).expanduser()

# If your actual paths differ, edit the run_dir strings directly here. Do not rely on mtime/order discovery.
AE_RUN_REGISTRY = {
    "mixed_baseline_raw_mse": {
        "run_dir": str(RUN_ROOT / "01_stage1_ae_pretraining/mixed_baseline_raw_mse"),
        "stage": "stage1a",
        "training_domain": "mixed",
        "test_domains": ["mixed", "pubmed", "nilearn", "neurovault"],
    },
    "mixed_balanced_raw_mse": {
        "run_dir": str(RUN_ROOT / "01_stage1_ae_pretraining/mixed_balanced_raw_mse"),
        "stage": "stage1a",
        "training_domain": "mixed",
        "test_domains": ["mixed", "pubmed", "nilearn", "neurovault"],
    },
    "mixed_balanced_hybrid_loss": {
        "run_dir": str(RUN_ROOT / "01_stage1_ae_pretraining/mixed_balanced_hybrid_loss"),
        "stage": "stage1a",
        "training_domain": "mixed",
        "test_domains": ["mixed", "pubmed", "nilearn", "neurovault"],
    },
    "mixed_to_pubmed": {
        "run_dir": str(RUN_ROOT / "02_stage1b_domain_finetune/pubmed/mixed_to_pubmed"),
        "stage": "stage1b",
        "training_domain": "pubmed",
        "test_domains": ["pubmed"],
        "cross_domain_test_domains": ["mixed", "nilearn", "neurovault"],
    },
    "mixed_to_nilearn": {
        "run_dir": str(RUN_ROOT / "02_stage1b_domain_finetune/nilearn/mixed_to_nilearn"),
        "stage": "stage1b",
        "training_domain": "nilearn",
        "test_domains": ["nilearn"],
        "cross_domain_test_domains": ["mixed", "pubmed", "neurovault"],
    },
    "mixed_to_neurovault": {
        "run_dir": str(RUN_ROOT / "02_stage1b_domain_finetune/neurovault/mixed_to_neurovault"),
        "stage": "stage1b",
        "training_domain": "neurovault",
        "test_domains": ["neurovault"],
        "cross_domain_test_domains": ["mixed", "pubmed", "nilearn"],
    },
}

# Notebook 6 sometimes used mixed_baseline_to_<domain>; uncomment these if that matches your output dirs.
# AE_RUN_REGISTRY["mixed_to_pubmed"]["run_dir"] = str(RUN_ROOT / "02_stage1b_domain_finetune/pubmed/mixed_baseline_to_pubmed")
# AE_RUN_REGISTRY["mixed_to_nilearn"]["run_dir"] = str(RUN_ROOT / "02_stage1b_domain_finetune/nilearn/mixed_baseline_to_nilearn")
# AE_RUN_REGISTRY["mixed_to_neurovault"]["run_dir"] = str(RUN_ROOT / "02_stage1b_domain_finetune/neurovault/mixed_baseline_to_neurovault")

TEST_JSONL = os.environ.get("NEUROVLM_TEST_JSONL", "")  # leave blank to use the repo fixed test split
OUTPUT_ROOT = Path(os.environ.get("NEUROVLM_AE_EVAL_OUTPUT_ROOT", REPO_DIR / "experiments/3dcnn")).expanduser()

EVAL_BATCH_SIZE = int(os.environ.get("NEUROVLM_AE_EVAL_BATCH_SIZE", "32"))
EVAL_NUM_WORKERS = int(os.environ.get("NEUROVLM_EVAL_NUM_WORKERS", "0"))
OVERWRITE = False
MAKE_QUALITATIVE_PLOTS = True
EVALUATE_STAGE1B_CROSS_DOMAIN = True

AE_RUN_REGISTRY

In [ ]:
cfg = EvaluationConfig(
    registry=AE_RUN_REGISTRY,
    output_root=OUTPUT_ROOT,
    test_jsonl=Path(TEST_JSONL).expanduser() if TEST_JSONL else None,
    device="auto",
    eval_batch_size=EVAL_BATCH_SIZE,
    num_workers=EVAL_NUM_WORKERS,
    overwrite=OVERWRITE,
    make_qualitative_plots=MAKE_QUALITATIVE_PLOTS,
    evaluate_stage1b_cross_domain=EVALUATE_STAGE1B_CROSS_DOMAIN,
)

EVAL_OUTPUT_DIR = run_evaluation(cfg)
EVAL_OUTPUT_DIR

In [ ]:
import json

selected_path = EVAL_OUTPUT_DIR / "04_final_selection/selected_stage2_checkpoints.json"
with selected_path.open() as f:
    selected = json.load(f)

print("Selected checkpoint manifest:", selected_path)
selected